In [1]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd

import learnQ as lq
import morphData as morph


(CVXPY) Apr 01 08:15:22 PM: Encountered unexpected exception importing solver DIFFCP:
ImportError('diffcp >= 1.0.15 is required')


# EBM Factor Model Experiment

### Example of how the data is being re-shaped (in morph) into the form required by learnQ function

In [2]:
np.random.seed(215)

T = 4
K = 2
N_minus_1 = 2

print("Original EBM data:\n")
# T x K
out_treated = np.random.rand(T, K)
print("out_treated:\n", out_treated)

# ((N-1)*T) x K  — this is what EBM's code produces
out_control = np.random.rand(N_minus_1 * T, K)
print("\nout_control:\n", out_control)

# Reshape control to (T, N-1, K) then transpose to (T, K, N-1)
out_control_3d = out_control.reshape(T, N_minus_1, K).transpose(0, 2, 1)  

# Add unit dimension to treated and concatenate
out_trt_3d = out_treated[:, np.newaxis, :].transpose(0, 2, 1)  

Y = np.concatenate([out_trt_3d, out_control_3d], axis=2)  

print("\nFinal Reshaped data:\n",Y)

Original EBM data:

out_treated:
 [[0.15392159 0.48956512]
 [0.18198889 0.20908939]
 [0.22316718 0.08422693]
 [0.91593932 0.60770076]]

out_control:
 [[0.30851461 0.74596553]
 [0.81033323 0.39261118]
 [0.41442981 0.17074698]
 [0.76798541 0.40117711]
 [0.3622311  0.40013476]
 [0.28704036 0.0098913 ]
 [0.33164009 0.22417675]
 [0.46732337 0.56413055]]

Final Reshaped data:
 [[[0.15392159 0.30851461 0.81033323]
  [0.48956512 0.74596553 0.39261118]]

 [[0.18198889 0.41442981 0.76798541]
  [0.20908939 0.17074698 0.40117711]]

 [[0.22316718 0.3622311  0.28704036]
  [0.08422693 0.40013476 0.0098913 ]]

 [[0.91593932 0.33164009 0.46732337]
  [0.60770076 0.22417675 0.56413055]]]


Now we reshape (using morph) the data produced by EBM's code and run the learnQ function on the training data:

In [ ]:
treated_data = pd.read_csv('treated_data.csv')
control_data = pd.read_csv('control_data.csv')

train_target_vectors, train_covariate_matrices, test_target_vector, test_covariate_matrix = morph.morph(treated_data, control_data)

Q_final, w_final = lq.learnQ(train_target_vectors, train_covariate_matrices, 10, 800, 1, 10, False)

print(w_final.round(2))


Step    0 | Loss: 2556.43393128
Step    0 | Loss: 2556.43393128 | w: [0.153 0.108 0.034 0.021 0.116 0.149 0.126 0.118 0.14  0.036]
Grad norm: 642.51298863
Step  200 | Loss: 467.38634398
Step  200 | Loss: 467.38634398 | w: [0.053 0.102 0.096 0.124 0.08  0.1   0.197 0.09  0.052 0.107]
Grad norm: 23.27386666


Examine the results:

In [ ]:
# get the learnQ prediction at the last time point.
pred_learnQ = test_covariate_matrix.numpy()@ Q_final @ w_final  # (K,)

# get the EBM prediction at the last time point
w_avg = pd.read_csv('T_0=40_K=10_w_avg.csv').to_numpy()
pred_avg = (test_covariate_matrix @ w_avg).squeeze()

# compare the synthetic covariates to the original
print("Original Covariates (truncated):\n", test_covariate_matrix.numpy().round(1)[:3])
print("Synthetic Covariates:\n",(test_covariate_matrix.numpy()@ Q_final).round(1))

# examine the learnQ weights
print("final learnQ weights:\n", w_final.round(2))

# the discrepancies between the true and the predicted
print("LearnQ discrepancy:\n", np.sum((test_target_vector.numpy() - pred_learnQ)**2).round(2))
print("avg_weights discrepancy:\n", np.sum((test_target_vector.numpy() - pred_avg.numpy())**2).round(2))

# the predictions
print("test_target_vector:\n", test_target_vector.numpy().round(2))
print("pred_avg:\n", pred_avg.numpy().T.round(2))
print("pred_learnQ:\n", pred_learnQ.round(2))


# VALIDATE THIS 
# something to note is that the model_t1 data is the data at time T+1, and is noiseless. This is how EBM evaluates his model.
# need to think about how to make apples to apples comparison. 
# I will have to compare my predictions also to the model_t1
# this means making a prediction based on the noiseless model_t1 data.
# have to investigate how EBM calculates the bias.

# Is it possible my method works better for data with fewer units?

Original Covariates (truncated):
 [[ 4.8  3.5  4.1  3.8  3.8  5.   5.8  4.2  5.1  5.9  4.4  3.4  4.2  2.8
   3.3  4.5  3.9  2.2  3.8  3.3  1.9  3.3  2.4  3.1  0.7  4.7  2.1  3.
   2.   3.7  2.7  2.8  1.6  1.4  1.9  2.   0.2  2.3  1.5  0.2  3.5  0.7
   0.5  0.7  0.5  2.2 -0.   1.3 -1.5]
 [ 5.5  3.7  3.6  5.   3.9  4.   4.5  3.3  6.1  4.2  5.2  2.6  4.6  3.7
   2.9  4.7  4.5  3.7  3.4  1.6  4.3  1.8  2.2  3.3  2.1  4.6  2.5  2.4
   4.7  3.   2.1  2.5  2.   1.2  2.2  2.5  0.4  2.   2.2  2.   0.9  1.7
   0.1  1.6  1.9  0.7  2.6  1.8  1.6]
 [ 4.2  2.7  4.9  3.4  5.8  3.9  3.9  5.3  4.2  3.8  3.8  4.4  4.2  5.9
   3.2  4.2  3.3  2.9  3.4  4.   4.1  3.3  2.1  5.1  5.1  2.6 -0.2  1.3
   2.8  2.9  1.9  3.4  2.3  3.6  1.3  3.2  2.6  4.8 -0.4  3.1  0.9  1.3
   1.5  1.1  0.5  0.4  0.3 -1.5  1.3]]
Synthetic Covariates:
 [[  5.2   5.7   1.7   5.2   6.2   5.    2.7   4.5 -12.7   4. ]
 [  4.1   6.6   0.9   5.7   5.7   6.1   5.1   5.5 -11.7   4.2]
 [  4.9   6.1   1.3   7.9   6.3   4.3   5.1   7.6  -9. 